# Jour 3 — Traitement de données et restitution

*Formation Python pour l'analyse de données — sujet*

**Paliers** : ★ indispensable — ★★ à faire si le ★ est fini — ★★★ pour aller plus loin.
Faites les ★ de tous les exercices avant d'attaquer les ★★★.

Réflexes sans assistant IA : `Tab` (autocomplétion), `Shift+Tab` (signature), `objet?` (aide),
`help(objet)`, `dir(objet)`, la doc officielle (https://docs.python.org/fr/3/, https://pandas.pydata.org/docs/)
et les aide-mémoire fournis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DONNEES = Path("donnees")
options = dict(sep=";", decimal=",", encoding="cp1252", dayfirst=True)
clients = pd.read_csv(DONNEES / "clients.csv", dtype={"code_postal": "string"},
                      parse_dates=["date_naissance", "date_creation"], **options)
contrats = pd.read_csv(DONNEES / "contrats.csv", parse_dates=["date_effet", "date_resiliation"], **options)
sinistres = pd.read_csv(DONNEES / "sinistres.csv", parse_dates=["date_survenance", "date_declaration"], **options)
print(len(clients), len(contrats), len(sinistres))

### Exercice 1 — Jointures  ★☆☆

Construisez `sin_complet` : chaque sinistre avec les informations de son contrat
(`produit`, `date_effet`, `date_resiliation`, `id_client`) puis du client (`nom`, `code_postal`).
- Quelle valeur de `how` pour garder **tous** les sinistres ?
- Vérifiez que le nombre de lignes n'a pas changé (sinon : clé en double à droite !) ;
  ★★ utilisez `validate="m:1"` et `indicator=True`.

In [ ]:
# À vous

### Exercice 2 — Agrégations et tableaux croisés  ★★☆

1. Par produit : nombre de sinistres, montant estimé moyen, montant réglé total,
   taux de rejet (part des sinistres `REJETE`) — avec l'agrégation nommée
   `groupby(...).agg(nom=(colonne, fonction))`.
2. `pivot_table` : nombre de sinistres par produit (lignes) et année de survenance (colonnes).

In [ ]:
# À vous

### Exercice 3 — Dates et séries temporelles  ★★☆

1. Ajoutez `delai_declaration` (jours entre survenance et déclaration) ;
   statistiques descriptives (`describe`) ; combien de délais négatifs ?
2. Sinistres survenus **hors période de garantie** (avant l'effet ou après la résiliation).
3. Nombre de sinistres par mois (`resample("MS")` sur un index de dates, ou `dt.to_period("M")`),
   puis moyenne glissante sur 3 mois (`rolling`).

In [ ]:
# À vous

### Exercice 4 — apply ou vectorisation ?  ★★☆

On veut une colonne `gravite` : `"grave"` si montant estimé > 5000, `"moyen"` si > 1000,
sinon `"faible"`. Écrivez-la avec `apply` + une fonction, puis avec `np.select`.
Comparez les temps avec `%timeit`. Conclusion ?

In [ ]:
# À vous

### Exercice 5 — Graphiques  ★☆☆

1. Matplotlib : diagramme en barres du nombre de sinistres par produit,
   avec titre et libellés d'axes.
2. Courbe du nombre de sinistres par mois (réutilisez `par_mois`).
3. Plotly Express : histogramme interactif des délais de déclaration, coloré par produit.

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

# À vous

## Exercice complet : notebook de contrôle qualité de bout en bout

Objectif : un module `controles_qualite.py` **réutilisable** + ce notebook qui l'appelle.

Chaque contrôle est une fonction qui reçoit les tables et renvoie un DataFrame des lignes
en anomalie avec deux colonnes communes : `table` et `controle`. Une fonction
`executer_controles` les lance tous et concatène les résultats.

Paliers :
- ★ 3 contrôles (doublons stricts, contrats orphelins, primes négatives) + synthèse ;
- ★★ tous les contrôles vus depuis hier + graphique de synthèse + export CSV ;
- ★★★ un dictionnaire de configuration (seuils, motifs) passé en paramètre, docstrings, et
  un test de chaque contrôle sur un petit DataFrame construit à la main.

### Exercice 6 — Module de contrôles  ★★★

Complétez le module (les deux premiers contrôles sont donnés comme modèles).

In [ ]:
%%writefile controles_qualite.py
"""Contrôles qualité sur les tables clients / contrats / sinistres."""
import pandas as pd


def _marquer(lignes, table, controle):
    return lignes.assign(table=table, controle=controle)


def doublons_stricts(tables):
    return pd.concat([_marquer(df[df.duplicated()], nom, "ligne dupliquée")
                      for nom, df in tables.items()])


def contrats_orphelins(tables):
    contrats, clients = tables["contrats"], tables["clients"]
    return _marquer(contrats[~contrats["id_client"].isin(clients["id_client"])],
                    "contrats", "client inexistant")


# À vous : primes_negatives, num_contrat_invalide, emails_invalides,
#          resiliation_avant_effet, declaration_avant_survenance, sinistres_orphelins...

CONTROLES = [doublons_stricts, contrats_orphelins]


def executer_controles(tables, controles=CONTROLES):
    """Lance tous les contrôles et renvoie les anomalies concaténées."""
    return pd.concat([controle(tables) for controle in controles], ignore_index=True)

In [ ]:
import importlib
import controles_qualite as cq
importlib.reload(cq)

tables = {"clients": clients, "contrats": contrats, "sinistres": sinistres}
anomalies = cq.executer_controles(tables)
synthese = anomalies.groupby(["table", "controle"]).size().rename("nb").reset_index()
synthese

### Exercice 7 — Synthèse, graphique, export  ★★☆

À partir de `synthese` et `anomalies` :
1. graphique en barres horizontales du nombre d'anomalies par contrôle, coloré par table ;
2. export de `anomalies` en CSV dans `sortie/` (un fichier par table, nommé avec la date du jour).

In [ ]:
# À vous

## Restitution interactive

Les *widgets* rendent un notebook explorable sans toucher au code.
(Si les widgets ne s'affichent pas au datalab, l'extension `ipywidgets` n'y est pas activée.)

### Exercice 8 — Un filtre interactif  ★★☆

Avec `ipywidgets.interact`, affichez le nombre de sinistres par mois pour le produit
choisi dans une liste déroulante.

In [ ]:
from ipywidgets import interact

# À vous

### Pour aller plus loin : Dash

Dash construit un vrai tableau de bord web (serveur Python) à partir de graphiques Plotly.
Il faut pouvoir lancer un serveur depuis le datalab : à vérifier avec votre DSI.

```python
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

app = Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(sorted(sin_complet["produit"].dropna().unique()), "AUTO", id="produit"),
    dcc.Graph(id="graphique"),
])

@app.callback(Output("graphique", "figure"), Input("produit", "value"))
def maj(produit):
    d = sin_complet[sin_complet["produit"] == produit]
    return px.histogram(d, x="date_survenance", title=f"Sinistres {produit}")

app.run(debug=True)
```

## Bilan

Reprenez le tableau « fil rouge » du jour 1 : pour un de vos programmes SAS, écrivez
le notebook Python équivalent en réutilisant `controles_qualite.py`.